In this notebook, we will merge the yamanishi datasets: enzymes, ion channels, GPCRs, and nuclear receptors into a single CSV file.

We will additionaly merge the drugbank dataset with Yamanishi, as they imply on doing the same thing in their paper.



In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR_YAMANISHI = Path("../data/positive dti datasets/yamanishi/final yamanishi") 
DATA_DIR_DRUGBANK = Path("../data/positive dti datasets/drugbank/drugbank beginning dataset") 

files_yamanishi = {
     "enzymes.csv",
    "gpcr.csv",
    "ion_channel.csv",
    "nuclear_receptor.csv",
}

#get the dataframes from yamanishi
dataframes_yamanishi = []
for file in files_yamanishi:
    csv_path = DATA_DIR_YAMANISHI / file
    df = pd.read_csv(csv_path)
    dataframes_yamanishi.append(df) 
#merge yamanishi dataframes
merged_yamanishi_df = pd.concat(dataframes_yamanishi, ignore_index=True)


merged_yamanishi_df.shape



(5127, 3)

In [2]:
#getting rid of drug target duplicates in yamanishi
merged_yamanishi_df = merged_yamanishi_df.drop_duplicates(subset=['drug_id','target_id'])
merged_yamanishi_df.shape
merged_yamanishi_df.columns
merged_yamanishi_df.head()

#save the merged yamanishi dataframe as a csv
merged_yamanishi_csv_path = DATA_DIR_YAMANISHI / "yamanishi_merged.csv"
merged_yamanishi_df.to_csv(merged_yamanishi_csv_path, index=False)

In [28]:

#get df drugbank
csv_path = DATA_DIR_DRUGBANK / "drugbank_dti_with_kegg.csv"
df_drugbank = pd.read_csv(csv_path)

df_drugbank.shape
df_drugbank.columns

Index(['drugbank_id', 'drug_name', 'smiles', 'kegg_drug_id',
       'protein_uniprot_id', 'kegg_protein_id', 'protein_sequence', 'organism',
       'label'],
      dtype='object')

In [29]:
#we wanna delete all rows in drugbank that have invalid SMILES and invalid sequences
import pandas as pd
from rdkit import Chem
from Bio.SeqUtils import IUPACData
from tqdm import tqdm


def is_valid_smiles(smiles):
    if not isinstance(smiles, str) or smiles.strip() == "":
        return False
    mol = Chem.MolFromSmiles(smiles)
    return mol is not None
VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

def is_valid_protein(seq):
    if not isinstance(seq, str) or seq.strip() == "":
        return False
    seq = seq.upper().strip()
    
    # Remove FASTA header if present
    if seq.startswith(">"):
        seq = "\n".join(seq.split("\n")[1:])
    
    seq = seq.replace("\n", "").replace(" ", "")
    
    # Reject if too short to be real protein
    if len(seq) < 20:
        return False

    return all(aa in VALID_AA for aa in seq)

tqdm.pandas()

df_drugbank["valid_smiles"] = df_drugbank["smiles"].progress_apply(is_valid_smiles)
df_drugbank["valid_protein"] = df_drugbank["protein_sequence"].progress_apply(is_valid_protein)

print("Before cleaning:", df_drugbank.shape)




  8%|▊         | 2060/24525 [00:00<00:02, 10369.51it/s][13:26:34] Explicit valence for atom # 0 N, 4, is greater than permitted
[13:26:34] Explicit valence for atom # 0 N, 4, is greater than permitted
[13:26:34] Explicit valence for atom # 0 N, 4, is greater than permitted
[13:26:34] Explicit valence for atom # 0 N, 4, is greater than permitted
 34%|███▍      | 8296/24525 [00:00<00:01, 9966.78it/s] [13:26:35] Explicit valence for atom # 13 Cl, 5, is greater than permitted
[13:26:35] Explicit valence for atom # 13 Cl, 5, is greater than permitted
[13:26:35] Explicit valence for atom # 13 Cl, 5, is greater than permitted
 50%|█████     | 12301/24525 [00:01<00:01, 9659.03it/s][13:26:35] Explicit valence for atom # 0 O, 3, is greater than permitted
[13:26:35] Explicit valence for atom # 3 N, 4, is greater than permitted
[13:26:35] Unusual charge on atom 0 number of radical electrons set to zero
[13:26:35] Explicit valence for atom # 4 F, 2, is greater than permitted
[13:26:35] Explicit val

Before cleaning: (24525, 11)


In [30]:
df_drugbank = df_drugbank[
    (df_drugbank["valid_smiles"] == True) &
    (df_drugbank["valid_protein"] == True)
].copy()

print("After cleaning:", df_drugbank.shape)


After cleaning: (24451, 11)


In [31]:
df_drugbank = df_drugbank.drop(columns=["valid_smiles", "valid_protein"])


In [33]:
print("Unique drugs after cleaning:", df_drugbank["drugbank_id"].nunique())
print("Unique proteins after cleaning:", df_drugbank["protein_uniprot_id"].nunique())

#also count unique pairs of drug and protein
print("Unique drug-protein pairs after cleaning:", df_drugbank[["drugbank_id", "protein_uniprot_id"]].drop_duplicates().shape[0])   

Unique drugs after cleaning: 8407
Unique proteins after cleaning: 4822
Unique drug-protein pairs after cleaning: 23883


In [34]:
# we want to keep only the unique drug protein pairs
df_drugbank = df_drugbank.drop_duplicates(subset=['drugbank_id','protein_uniprot_id'])
df_drugbank.shape

(23883, 9)

In [ ]:
#We will now merge drugbank with yamanishi

print("DrugBank DTI dataset head:", df_drugbank.head())
print("Yamanishi merged DTI dataset head:", merged_yamanishi_df.head())


In [36]:
#yamanishi columns are named wrongly, we will rename them


merged_yamanishi_df = merged_yamanishi_df.rename(columns={
    "drug_id": "kegg_protein_id",
    "target_id": "kegg_drug_id"
})


In [37]:
#make sure the KEGG columns are clean
df_drugbank["kegg_drug_id"] = df_drugbank["kegg_drug_id"].astype(str).str.strip()
df_drugbank["kegg_protein_id"] = df_drugbank["kegg_protein_id"].astype(str).str.strip()

merged_yamanishi_df["kegg_drug_id"] = merged_yamanishi_df["kegg_drug_id"].astype(str).str.strip()
merged_yamanishi_df["kegg_protein_id"] = merged_yamanishi_df["kegg_protein_id"].astype(str).str.strip()


In [38]:
#check if df_drugbank["kegg_protein_id"] is all NaN
df_drugbank["kegg_protein_id"].unique() 
df_drugbank["kegg_drug_id"].unique() 

#check if there is any kegg drug id in drugbank that is nan
df_drugbank[df_drugbank["kegg_drug_id"].isna()]

,drugbank_id,drug_name,smiles,kegg_drug_id,protein_uniprot_id,kegg_protein_id,protein_sequence,organism,label


In [39]:
#since kegg_protein_id in drugbank is all NaN 
# we will transform kegg_protein_id in merged_yamanishi_df to uniprot ids
import pandas as pd
import requests
from tqdm import tqdm


def kegg_to_uniprot(kegg_ids, species="hsa"):
    """
    Maps KEGG gene/protein IDs to UniProt IDs using KEGG REST API.
    
    Example input: ["hsa:1956", "hsa:5290"]
    """
    base_url = f"https://rest.kegg.jp/conv/uniprot/{species}"
    
    response = requests.get(base_url)
    
    if response.status_code != 200:
        raise Exception("Failed to fetch KEGG-UniProt mapping table")

    mapping = {}
    
    for line in response.text.strip().split("\n"):
        kegg, uniprot = line.split("\t")
        kegg_id = kegg.replace("gene:", "")
        uniprot_id = uniprot.replace("uniprot:", "")
        mapping[kegg_id] = uniprot_id

    # Map only requested ids
    return {kid: mapping.get(kid, None) for kid in kegg_ids}


In [40]:
# Get unique KEGG protein IDs
kegg_protein_ids = merged_yamanishi_df["kegg_protein_id"].dropna().unique().tolist()

print("Total unique KEGG protein IDs:", len(kegg_protein_ids))


Total unique KEGG protein IDs: 989


In [41]:
kegg_uniprot_map = kegg_to_uniprot(kegg_protein_ids, species="hsa")

print("Example mappings:")
list(kegg_uniprot_map.items())[:10]


Example mappings:


[('hsa:190', 'up:F1D8P4'),
 ('hsa:2099', 'up:G4XH65'),
 ('hsa:2100', 'up:Q7LCB3'),
 ('hsa:2101', 'up:Q569H8'),
 ('hsa:2103', 'up:O95718'),
 ('hsa:2104', 'up:F1D8R5'),
 ('hsa:2908', 'up:F1D8N4'),
 ('hsa:3174', 'up:Q14541'),
 ('hsa:367', 'up:Q9NUA2'),
 ('hsa:4306', 'up:B0ZBF6')]

In [42]:
merged_yamanishi_df["uniprot_id"] = merged_yamanishi_df["kegg_protein_id"].map(kegg_uniprot_map)

# Check how many mapped successfully
print("Mapped UniProt IDs:", merged_yamanishi_df["uniprot_id"].notna().sum())
print("Unmapped IDs:", merged_yamanishi_df["uniprot_id"].isna().sum())


Mapped UniProt IDs: 5124
Unmapped IDs: 3


In [43]:
df_drugbank = df_drugbank.rename(columns={
    "protein_uniprot_id": "uniprot_id"
})

df_drugbank = df_drugbank[[
    "kegg_drug_id",
    "uniprot_id",
    "label"
]]


In [44]:
yamanishi_clean = merged_yamanishi_df.rename(columns={
    "drug_id": "kegg_drug_id"
})

yamanishi_clean = yamanishi_clean[[
    "kegg_drug_id",
    "uniprot_id"
]]

# Yamanishi is assumed positive interactions
yamanishi_clean["label"] = 1



In [45]:
yamanishi_clean = yamanishi_clean.dropna(subset=["kegg_drug_id", "uniprot_id"])
df_drugbank = df_drugbank.dropna(subset=["kegg_drug_id", "uniprot_id"])
yamanishi_clean.shape, df_drugbank.shape

((5124, 3), (23883, 3))

In [46]:
union = pd.concat([
    df_drugbank,
    yamanishi_clean
], ignore_index=True)

# Deduplicate based on standardized IDs
union = union.drop_duplicates(subset=["kegg_drug_id", "uniprot_id"])

print("Final Union Dataset Shape:", union.shape)


Final Union Dataset Shape: (21053, 3)


In [47]:
# now we check the shapes

print("DrugBank DTI dataset shape:", df_drugbank.shape)
print("Yamanishi merged DTI dataset shape:", merged_yamanishi_df.shape)
print("Merged DrugBank and Yamanishi DTI dataset shape:", union.shape)

union.head()


DrugBank DTI dataset shape: (23883, 3)
Yamanishi merged DTI dataset shape: (5127, 4)
Merged DrugBank and Yamanishi DTI dataset shape: (21053, 3)


,kegg_drug_id,uniprot_id,label
0,D03136,P00734,1
1,D00573,P22888,1
2,D00573,P01148,1
3,D00573,P30968,1
4,D04369,P0AC13,1


In [48]:
print("Unique drugs:", union["kegg_drug_id"].nunique())
print("Unique proteins:", union["uniprot_id"].nunique())
print("Positive interactions:", union["label"].sum())


Unique drugs: 3239
Unique proteins: 5809
Positive interactions: 21053


In [49]:
#Find out the unique drugs and proteins in the each of the datasets
unique_drugs_drugbank = df_drugbank["kegg_drug_id"].nunique()
unique_proteins_drugbank = df_drugbank["uniprot_id"].nunique()
unique_drugs_yamanishi = yamanishi_clean["kegg_drug_id"].nunique()
unique_proteins_yamanishi = yamanishi_clean["uniprot_id"].nunique()
print("DrugBank - Unique Drugs:", unique_drugs_drugbank, "Unique Proteins:", unique_proteins_drugbank)
print("Yamanishi - Unique Drugs:", unique_drugs_yamanishi, "Unique Proteins:", unique_proteins_yamanishi)

DrugBank - Unique Drugs: 2579 Unique Proteins: 4822
Yamanishi - Unique Drugs: 791 Unique Proteins: 987


In [50]:
# count the duplicated rows for the union dataset
duplicated_rows = union.duplicated(subset=["kegg_drug_id", "uniprot_id"]).sum()
print("Number of duplicated drug-protein pairs in the union dataset:", duplicated_rows)

Number of duplicated drug-protein pairs in the union dataset: 0


**From the paper:**

The number of unique drugs in our **positive datasets** is **2,118**,comprising 
**1,328 from DrugBank and 790 from Yamanishi’s data**. In the same data, the
number of unique **drug-targets is 2,077 (706 from DrugBank and 1,371 from Yamanishi ).**
Finally, the number of known DTIs between the drugs and targets in the positive data is
**10,736 (3,530 from DrugBank and 7,206 from Yamanishi.)**

In [52]:
#convert the union into csv
DATA_DIR_POSITIVE_DTI = Path("../data/positive dti datasets")
union_csv_path = DATA_DIR_POSITIVE_DTI / "drugbank_yamanishi_merged.csv"
union.to_csv(union_csv_path, index=False)
